# "THE POUR IS RIGHT" - Wine Pricer Capstone

## Week 8 Order of play

DAY 1: Modal and the SpecialistAgent  
DAY 2: RAG with Qdrant, FrontierAgent, Ensemble  
DAY 3: ScannerAgent, MessagingAgent  
DAY 4: The planning agent  
DAY 5: The user interface  

## DAY 5: the app

The last step is a user interface: a web page that runs the planner on a schedule and shows the deals it has found. We build it with Gradio, piece by piece, then launch the finished app. This follows `week8/day5.ipynb`.

The pieces:

- `deal_agent_framework.py`: opens the vector store, keeps the memory of deals already surfaced in `memory.json`, and runs the planner. It uses the LangGraph planner; change `PLANNER = "langgraph"` to `"fixed"` to use Ed's fixed workflow instead.
- `pour_is_right.py`: the Gradio app, adapted from Ed's `price_is_right.py`.

## Two changes from Ed's app

- **It looks for deals once an hour, not every 5 minutes.** Our shops change their offers about once a day, and each run costs about 5 cents plus waking the model on Modal. A **"Look for deals now"** button runs it whenever you like.
- **A logging fix.** Ed's app added a new log handler on every run and never removed it, so after a few runs each line appeared several times in the log panel.

In [ ]:
import gradio as gr
from agents.deals import Opportunity, Deal

# Build the interface step by step

Gradio describes a page as a set of nested blocks. `gr.Row()` lays its contents side by side, and `gr.Markdown` shows text.

Each `launch()` below opens a small page in your browser. They are separate demo pages: you can close the browser tabs after looking at them.

In [ ]:
with gr.Blocks(title="The Pour Is Right", fill_width=True) as ui:

    with gr.Row():
        gr.Markdown('<div style="text-align: center;font-size:24px">The Pour Is Right - Deal Hunting Agentic AI</div>')
    with gr.Row():
        gr.Markdown('<div style="text-align: center;font-size:14px">Autonomous agents that find online wine deals, working with a fine-tuned LLM on Modal and a RAG pipeline with Claude and Qdrant.</div>')

ui.launch(inbrowser=True)

## Add a table of deals

`gr.State` holds data that belongs to the page but isn't shown, here the list of deals. `ui.load` runs a function when the page opens; it turns the deals into rows for `gr.Dataframe`.

We start with one made-up deal so there is something to show.

In [ ]:
with gr.Blocks(title="The Pour Is Right", fill_width=True) as ui:

    initial_deal = Deal(product_description="Example Napa Cabernet Sauvignon", price=25.0, currency="USD", url="https://www.example.com")
    initial_opportunity = Opportunity(deal=initial_deal, estimate=60.0, discount=35.0)
    opportunities = gr.State([initial_opportunity])

    def get_table(opps):
        return [[opp.deal.product_description, opp.deal.price, opp.estimate, opp.discount, opp.deal.url] for opp in opps]

    with gr.Row():
        gr.Markdown('<div style="text-align: center;font-size:24px">The Pour Is Right - Deal Hunting Agentic AI</div>')
    with gr.Row():
        gr.Markdown('<div style="text-align: center;font-size:14px">Deals surfaced so far:</div>')
    with gr.Row():
        opportunities_dataframe = gr.Dataframe(
            headers=["Description", "Price", "Estimate", "Discount", "URL"],
            wrap=True,
            column_widths=[4, 1, 1, 1, 2],
            row_count=10,
            col_count=5,
            max_height=400,
        )

    ui.load(get_table, inputs=[opportunities], outputs=[opportunities_dataframe])

ui.launch(inbrowser=True)

## Make a row clickable

`opportunities_dataframe.select(...)` runs a function when you click a row. Here, clicking the example row sends its alert to your Telegram, through the MessagingAgent from day 3. Clicking sends a real message.

In [ ]:
from agents.messaging_agent import MessagingAgent

messenger = MessagingAgent()

with gr.Blocks(title="The Pour Is Right", fill_width=True) as ui:

    initial_deal = Deal(product_description="Example Napa Cabernet Sauvignon", price=25.0, currency="USD", url="https://www.example.com")
    initial_opportunity = Opportunity(deal=initial_deal, estimate=60.0, discount=35.0)
    opportunities = gr.State([initial_opportunity])

    def get_table(opps):
        return [[opp.deal.product_description, opp.deal.price, opp.estimate, opp.discount, opp.deal.url] for opp in opps]

    def do_select(opportunities, selected_index: gr.SelectData):
        row = selected_index.index[0]
        opportunity = opportunities[row]
        messenger.alert(opportunity)

    with gr.Row():
        gr.Markdown('<div style="text-align: center;font-size:24px">The Pour Is Right - Deal Hunting Agentic AI</div>')
    with gr.Row():
        gr.Markdown('<div style="text-align: center;font-size:14px">Deals surfaced so far (click a row to send its alert):</div>')
    with gr.Row():
        opportunities_dataframe = gr.Dataframe(
            headers=["Description", "Price", "Estimate", "Discount", "URL"],
            wrap=True,
            column_widths=[4, 1, 1, 1, 2],
            row_count=10,
            col_count=5,
            max_height=400,
        )

    ui.load(get_table, inputs=[opportunities], outputs=[opportunities_dataframe])
    opportunities_dataframe.select(do_select, inputs=[opportunities], outputs=[])

ui.launch(inbrowser=True)

# The full app

`pour_is_right.py` adds three more pieces to what we just built:

- **A live log panel.** The planner runs in a background thread. Its log lines go into a queue, and a generator function takes them out and updates the page as they arrive, with each agent's colour.
- **A 3D map of the vector store**: 800 wines reduced to three dimensions with t-SNE, coloured by country, like the 2D chart on day 2.
- **A timer** that runs the planner once an hour, plus the button.

Read through `pour_is_right.py` and `deal_agent_framework.py` before running it. Most of the code is what you have already seen in these notebooks.

## Start with an empty memory

`memory.json` lists the deals already surfaced, so that the same one isn't sent twice. This empties it, so that today's deals can be found again.

In [ ]:
from deal_agent_framework import DealAgentFramework

DealAgentFramework.reset_memory()

## Run it

**The app runs the planner as soon as the page opens**, so a run starts straight away: about a minute and a half, about 5 cents, and probably one Telegram message. After that it runs once an hour, or when you press the button.

The cell keeps running for as long as the app is open. To stop the app, press the stop button on this cell (Interrupt).

Two things to know:

- **Close any other notebook that has the vector store open** before starting, such as day 2 or day 4. Only one program can use it at a time.
- If you leave it running, each hourly run costs about 5 cents. Stop it when you have finished.

In [ ]:
import sys

!{sys.executable} pour_is_right.py

# You've finished

From a raw scrape of wine reviews to an agent system that finds real deals and messages your phone:

| Week | What you built | Best result |
|---|---|---|
| 6 | Data curation, LLM rewriting, traditional ML, neural networks, frontier models | Claude Sonnet 5, \$9.06 |
| 7 | QLoRA fine-tuning of Qwen 3 4B on a free GPU | \$10.10, from \$23.15 before training |
| 8 | Modal serving, RAG with Qdrant, an ensemble, five agents, a LangGraph planner, this app | Ensemble, \$7.96 |

## Clean up on Modal

The fine-tuned model's files stay on a Modal volume, and storage is billed even when nothing runs. When you no longer need the app, delete the volume, and optionally stop the service:

```
uv run modal volume delete hf-hub-cache
uv run modal app stop wine-pricer-service
```

Or in the Modal dashboard: Storage, then Volumes, then `hf-hub-cache`, then Delete.